# LangChain: Q&A over Documents (Groq Llama 3.1)

## Overview
Learn how to build a question-answering system over your own documents using:
- **Document Loaders** - Load data from various sources
- **Embeddings** - Convert text to numerical vectors
- **Vector Stores** - Store and search embeddings
- **Retrieval QA Chain** - Combine retrieval + LLM for answers

**Model Used:** Groq Llama-3.1-8b-instant  
**Embeddings:** HuggingFace (sentence-transformers)

**Use Case:** Query a product catalog, company docs, or any text corpus


In [7]:
# Cell 2: Install Required Packages and Check Versions

!pip install langchain langchain-groq python-dotenv docarray sentence-transformers -q

# Check versions
import langchain
print(f"LangChain version: {langchain.__version__}")

# Test imports
try:
    from langchain_community.document_loaders import CSVLoader
    print("✅ CSVLoader import successful")
except ImportError as e:
    print(f"❌ CSVLoader import failed: {e}")

try:
    from langchain_community.vectorstores import DocArrayInMemorySearch
    print("✅ DocArrayInMemorySearch import successful")
except ImportError as e:
    print(f"❌ DocArrayInMemorySearch import failed: {e}")

try:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    print("✅ HuggingFaceEmbeddings import successful")
except ImportError as e:
    print(f"❌ HuggingFaceEmbeddings import failed: {e}")

try:
    from langchain.chains import RetrievalQA
    print("✅ RetrievalQA import successful")
except ImportError as e:
    print(f"❌ RetrievalQA import failed: {e}")


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


LangChain version: 1.2.8
✅ CSVLoader import successful
✅ DocArrayInMemorySearch import successful
✅ HuggingFaceEmbeddings import successful
❌ RetrievalQA import failed: No module named 'langchain.chains'


In [8]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
from dotenv import load_dotenv, find_dotenv
from IPython.display import display, Markdown

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings('ignore')

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")


✅ Environment loaded successfully


In [3]:
# Cell 4: Initialize Groq LLM

from langchain_groq import ChatGroq

# Set model
llm_model = "llama-3.1-8b-instant"

# Initialize LLM
llm = ChatGroq(temperature=0, model=llm_model, groq_api_key=groq_api_key)

print(f"✅ Model initialized: {llm_model}")


✅ Model initialized: llama-3.1-8b-instant


In [11]:
# Cell 5: Load Document Data

from langchain_community.document_loaders import CSVLoader

# Load the outdoor clothing catalog
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding='utf-8')

print(f"✅ CSV Loader created for: {file}")

✅ CSV Loader created for: OutdoorClothingCatalog_1000.csv


In [12]:
# Cell 6: Quick Start - Create Index and Query

# Note: VectorstoreIndexCreator is deprecated in LangChain 1.x
# We'll create the components manually instead

from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use HuggingFace embeddings (free, local)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Load documents
docs = loader.load()

# Create vector store index manually
db = DocArrayInMemorySearch.from_documents(docs, embeddings)

# Create a simple query function (replacing index.query)
def query_index(query, llm):
    # Get similar documents
    similar_docs = db.similarity_search(query, k=4)
    
    # Combine documents
    context = "\n\n".join([doc.page_content for doc in similar_docs])
    
    # Create prompt
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    
    # Get response from LLM
    response = llm.invoke(prompt)
    return response.content

# Store db and query function globally for later use
index = {"db": db, "query": query_index}

print("✅ Vector store index created successfully (manual implementation)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store index created successfully (manual implementation)


In [13]:
# Cell 7: Run Query on Index

# Define query
query = "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

# Query the index using our custom function
response = index["query"](query, llm)

# Display response
display(Markdown(response))

**Sun Protection Shirts**
| Name | Description | Fabric | Care | Features |
| --- | --- | --- | --- | --- |
| Women's Tropical Tee, Sleeveless | Five-star sleeveless button-up shirt with SunSmart protection | 71% nylon, 29% polyester | Machine wash and dry | UPF 50+, wrinkle resistant, low-profile pockets |
| Sun Shield Shirt by | High-performance sun shirt with SPF 50+ protection | 78% nylon, 22% Lycra Xtra Life fiber | Handwash, line dry | Moisture-wicking, abrasion resistant, UPF 50+ |
| Sunrise Tee | Lightweight, UV-protective button down shirt | 71% nylon, 29% polyester | Machine wash and dry | UPF 50+, wrinkle-free, moisture-wicking |
| Tropical Breeze Shirt | Lightweight, breathable long-sleeve UPF shirt | 71% nylon, 29% polyester | Machine wash and dry | UPF 50+, wrinkle-resistant, moisture-wicking |

**Summary of Each Shirt:**

1. **Women's Tropical Tee, Sleeveless**: A sleeveless button-up shirt with SunSmart protection, perfect for warm weather. It's wrinkle resistant, has low-profile pockets, and provides UPF 50+ protection.
2. **Sun Shield Shirt by**: A high-performance sun shirt with SPF 50+ protection, designed for water activities. It's moisture-wicking, abrasion resistant, and has a UPF 50+ rating.
3. **Sunrise Tee**: A lightweight, UV-protective button down shirt designed for hot weather. It's wrinkle-free, moisture-wicking, and has a UPF 50+ rating.
4. **Tropical Breeze Shirt**: A lightweight, breathable long-sleeve UPF shirt designed for outdoor activities. It's wrinkle-resistant, moisture-wicking, and has a UPF 50+ rating.

All of these shirts provide excellent sun protection with UPF 50+ ratings, making them perfect for outdoor activities and travel.

## Step-by-Step: Under the Hood

Now let's break down what's happening in the one-liner approach. We'll go through each step manually to understand the process.


In [14]:
# Cell 8: Step-by-Step - Load Documents

from langchain_community.document_loaders import CSVLoader

# Load documents
loader = CSVLoader(file_path=file, encoding='utf-8')
docs = loader.load()

print(f"✅ Loaded {len(docs)} documents")
print("\nFirst document:")
print(docs[0])

✅ Loaded 1000 documents

First document:
page_content=': 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. 

Size & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. 

Specs: Approx. weight: 1 lb.1 oz. per pair. 

Construction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. 

Questions? Please contact us for any inquiries.' metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}


In [15]:
# Cell 9: Create Embeddings

from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embeddings model loaded")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings model loaded


In [16]:
# Cell 10: Test Embeddings

# Create embedding for a sample text
embed = embeddings.embed_query("Hi my name is Harrison")

print(f"Embedding dimension: {len(embed)}")
print(f"\nFirst 5 values: {embed[:5]}")


Embedding dimension: 384

First 5 values: [-0.06155332177877426, -0.06207890436053276, -0.018953030928969383, 0.048299092799425125, -0.028553970158100128]


In [17]:
# Cell 11: Create Vector Database

from langchain_community.vectorstores import DocArrayInMemorySearch

# Create vector store from documents
db = DocArrayInMemorySearch.from_documents(
    docs, 
    embeddings
)

print(f"✅ Vector database created with {len(docs)} documents")


✅ Vector database created with 1000 documents


In [18]:
# Cell 12: Test Similarity Search

# Query for similar documents
query = "Please suggest a shirt with sunblocking"
docs = db.similarity_search(query)

print(f"Found {len(docs)} similar documents")
print("\nMost similar document:")
print(docs[0])


Found 4 similar documents

Most similar document:
page_content=': 255
name: Sun Shield Shirt by
description: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. 

Size & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.

Fabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.

Additional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.

Sun Protection That Won't Wear Off
Our high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun's harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective UV protectant.' metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 255}


In [19]:
# Cell 13: Create Retriever

# Convert vector store to retriever
retriever = db.as_retriever()

print("✅ Retriever created from vector store")


✅ Retriever created from vector store


In [20]:
# Cell 14: Manual Question Answering (Without Chain)

# Combine retrieved documents into single text
qdocs = "".join([docs[i].page_content for i in range(len(docs))])

# Create prompt with documents + question
response = llm.invoke(
    f"{qdocs}\n\nQuestion: Please list all your shirts with sun protection \
in a table in markdown and summarize each one."
)

# Display response
display(Markdown(response.content))


**Sun Protection Shirts**
==========================

| Name | Description | Fabric | UPF Rating | Care Instructions |
| --- | --- | --- | --- | --- |
| Sun Shield Shirt | High-performance sun shirt for blocking UV rays | 78% nylon, 22% Lycra Xtra Life | 50+ | Handwash, line dry |
| Tropical Breeze Shirt | Lightweight, breathable long-sleeve shirt for superior sun protection | 71% nylon, 29% polyester | 50+ | Machine wash and dry |
| Men's Plaid Tropic Shirt | Ultracomfortable sun protection shirt for fishing and travel | 52% polyester, 48% nylon | 50+ | Machine wash and dry |
| Women's Tropical Tee | Sleeveless button-up shirt with SunSmart protection | 71% nylon, 29% polyester | 50+ | Machine wash and dry |

**Summary of Each Shirt**

1. **Sun Shield Shirt**: A high-performance sun shirt that provides excellent protection from UV rays, suitable for outdoor activities.
2. **Tropical Breeze Shirt**: A lightweight, breathable long-sleeve shirt designed for fishing and travel, offering superior sun protection and comfort.
3. **Men's Plaid Tropic Shirt**: A comfortable sun protection shirt designed for fishing and travel, featuring wrinkle-resistant fabric and quick-drying technology.
4. **Women's Tropical Tee**: A sleeveless button-up shirt with SunSmart protection, designed for women who want to stay cool and protected from the sun.

All of these shirts offer UPF 50+ protection, which blocks 98% of the sun's harmful UV rays, making them suitable for outdoor activities and travel.

## RetrievalQA Chain

Instead of manually combining documents and querying the LLM, we can use the **RetrievalQA** chain to automate this process.

**Chain Types:**
- **stuff** - Put all docs into one prompt (simplest, works for small docs)
- **map_reduce** - Query each doc separately, then combine answers
- **refine** - Iteratively build answer across docs
- **map_rerank** - Score each doc's answer, return highest


In [ ]:
# Cell 16: Create RetrievalQA Chain

# Try different imports for RetrievalQA
try:
    from langchain.chains import RetrievalQA
    print("✅ Imported from langchain.chains")
except ImportError:
    try:
        from langchain.chains.retrieval_qa import RetrievalQA
        print("✅ Imported from langchain.chains.retrieval_qa")
    except ImportError:
        try:
            from langchain.chains.retrieval import RetrievalQA
            print("✅ Imported from langchain.chains.retrieval")
        except ImportError:
            print("❌ RetrievalQA not found, implementing manually")
            RetrievalQA = None

if RetrievalQA:
    # Create QA chain with "stuff" method
    qa_stuff = RetrievalQA.from_chain_type(
        llm=llm, 
        chain_type="stuff",  # Simple method: stuff all docs into one prompt
        retriever=retriever, 
        verbose=True
    )
    print("✅ RetrievalQA chain created")
else:
    # Manual implementation
    def qa_stuff_run(query):
        # Get relevant documents (limit to 2)
        docs = retriever.invoke(query)[:2]  # Limit to 2 documents
        # Combine documents
        context = "\n\n".join([doc.page_content for doc in docs])
        # Create prompt
        prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
        # Get response
        response = llm.invoke(prompt)
        return response.content
    
    qa_stuff = {"run": qa_stuff_run}
    print("✅ Manual QA implementation created")

❌ RetrievalQA not found, implementing manually
✅ Manual QA implementation created


In [28]:
# Cell 17: Run RetrievalQA Chain

# Define query
query = "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

# Run the chain (using our manual implementation)
response = qa_stuff["run"](query)

# Display response
display(Markdown(response))

Retrieved 2 documents
Context length: 1543 characters


**Sun Protection Shirts**
=========================

| Name | Description | Fabric | Care | Sun Protection |
| --- | --- | --- | --- | --- |
| Women's Tropical Tee, Sleeveless | Five-star sleeveless button-up shirt with SunSmart protection | 71% nylon, 29% polyester | Machine wash and dry | UPF 50+, blocks 98% of sun's harmful rays |
| Sun Shield Shirt by [Brand] | High-performance sun shirt for blocking UV rays | 78% nylon, 22% Lycra Xtra Life fiber | Handwash, line dry | UPF 50+, blocks 98% of sun's harmful rays |

**Summary:**

1. **Women's Tropical Tee, Sleeveless**: A sleeveless button-up shirt with a fitted design and SunSmart protection. It's made of a blend of nylon and polyester, easy to care for, and provides high-level sun protection.
2. **Sun Shield Shirt by [Brand]**: A high-performance sun shirt designed for comfort and protection. It's made of a blend of nylon and Lycra, requires handwashing and line drying, and provides excellent sun protection, recommended by The Skin Cancer Foundation.

In [29]:
# Cell 18: Compare Index Query vs RetrievalQA

# Both approaches produce similar results
print("Method 1: Index Query (one-liner)")
response1 = index["query"](query, llm)
print(response1[:200] + "..." if len(response1) > 200 else response1)

print("\nMethod 2: RetrievalQA Chain (step-by-step)")
response2 = qa_stuff["run"](query)
print(response2[:200] + "..." if len(response2) > 200 else response2)

print("\n✅ Both methods achieve the same result!")
print("   - Index Query: Fast prototyping")
print("   - RetrievalQA: More control and customization")

Method 1: Index Query (one-liner)
**Sun Protection Shirts**
| Name | Description | Fabric | Care | Features |
| --- | --- | --- | --- | --- |
| Women's Tropical Tee, Sleeveless | Five-star sleeveless button-up shirt with SunSmart prot...

Method 2: RetrievalQA Chain (step-by-step)
Retrieved 2 documents
Context length: 1543 characters
**Sun Protection Shirts**

| Name | Description | Fabric | Care | Sun Protection |
| --- | --- | --- | --- | --- |
| Women's Tropical Tee, Sleeveless | Five-star sleeveless b...

✅ Both methods achieve the same result!
   - Index Query: Fast prototyping
   - RetrievalQA: More control and customization


In [30]:
# Cell 19: Advanced - Custom Embeddings in Index

# You can customize the index creation with specific embeddings
custom_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create custom vector store
custom_db = DocArrayInMemorySearch.from_documents(docs, custom_embeddings)

print("✅ Custom index created with specified embeddings")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Custom index created with specified embeddings


In [33]:
# Cell 20: Test Different Queries

# Try different questions about the catalog
queries = [
    "What products are available for cold weather?",
    "Do you have any jackets with waterproof features?",
    "What's the price range of your products?"
]

for q in queries:
    print(f"\nQuery: {q}")
    print("-" * 60)
    response = qa_stuff["run"](q)
    print(response)
    print()


Query: What products are available for cold weather?
------------------------------------------------------------
Retrieved 2 documents
Context length: 1719 characters
Based on the provided information, the following products are available for cold weather:

1. FrostGuard Pro Clava: This ultra clava provides windproof protection for your face in extreme cold weather, making it ideal for skiing and long rides on windy lifts.

2. Arctic Wind Insulated Pullover: This insulated pullover hoodie features PrimaLoft insulation, providing warmth for light and moderate activities in transitional and cold weather. It's suitable for temperatures up to -5° for moderate activity.


Query: Do you have any jackets with waterproof features?
------------------------------------------------------------
Retrieved 2 documents
Context length: 1506 characters
Yes, we have several jackets with waterproof features. 

Our Kids' Trail Adventure Rain Jacket, Colorblock Lined (item 416) features a durable laminat

## Chain Types Comparison

### 1. **Stuff** (Used in this notebook)
- **How it works:** Combines all retrieved docs into one prompt
- **Pros:** Simple, cheap (one LLM call), works well for small docs
- **Cons:** Limited by LLM context window
- **Best for:** Small to medium document sets

### 2. **Map Reduce**
- **How it works:** Process each doc separately, then combine results
- **Pros:** Works with any number of docs, can parallelize
- **Cons:** More expensive (multiple LLM calls), treats docs independently
- **Best for:** Large document sets, parallel processing

### 3. **Refine**
- **How it works:** Iteratively build answer by processing docs sequentially
- **Pros:** Good for building comprehensive answers over time
- **Cons:** Slow (sequential), expensive, can be verbose
- **Best for:** Detailed, evolving answers

### 4. **Map Rerank**
- **How it works:** Score each doc's relevance, return best answer
- **Pros:** Fast (parallel), focused answers
- **Cons:** Requires good scoring prompts, multiple LLM calls
- **Best for:** Finding single best answer from many docs


In [34]:
# Cell 22: Try Map Reduce Chain Type

# Note: RetrievalQA with map_reduce is not available in LangChain 1.x
# Implementing a simple version manually

def qa_map_reduce_run(query):
    # Get relevant documents
    docs = retriever.invoke(query)[:4]  # Get more docs for "map reduce"
    
    # "Map" phase: Process each document separately
    individual_answers = []
    for doc in docs:
        prompt = f"Context: {doc.page_content}\n\nQuestion: {query}\n\nAnswer briefly:"
        response = llm.invoke(prompt)
        individual_answers.append(response.content)
    
    # "Reduce" phase: Combine answers
    combined_answers = "\n".join(individual_answers)
    final_prompt = f"Individual answers:\n{combined_answers}\n\nQuestion: {query}\n\nProvide a final comprehensive answer:"
    final_response = llm.invoke(final_prompt)
    return final_response.content

qa_map_reduce = {"run": qa_map_reduce_run}

# Test query
query = "Summarize all the jacket options available"
response = qa_map_reduce["run"](query)

print("\nMap Reduce Response:")
display(Markdown(response))


Map Reduce Response:


Based on the information provided, here's a comprehensive summary of the jacket options available:

1. **Storm Shield Field Jacket**:
   - Style: Slightly Fitted
   - Fabric: Ripstop blend of 98% cotton and 2% spandex
   - Features: Garment dyed, machine wash and dry care, feminine stitch detailing, two chest pockets, two lower cargo pockets, interior adjustable drawstring, and two buttons at cuffs.

2. **The Ultimate Hunter's Jacket**:
   - Can be worn in four ways:
     1. Alone
     2. Zipped into the Big Game System
     3. Zipped into the Gore-Tex Soft-Shell
     4. Compact and carried separately for emergency use.

3. **Cozy Core 3-in-1 Jacket**:
   - Offers two main configurations:
     1. Shell only (wind- and water-resistant)
     2. Shell with 100% sweater fleece liner (for added warmth)

4. **Adrenaline Wind Colorblock Jacket**:
   - This is the only specific jacket option mentioned, but its features and configurations are not detailed.

In summary, there are three main jacket options with varying features and configurations, and one specific jacket option with unknown features.

## Summary: Q&A over Documents

### Key Components:

1. **Document Loaders** - Import data (CSV, PDF, web, etc.)
2. **Embeddings** - Convert text to numerical vectors (semantic meaning)
3. **Vector Store** - Store embeddings for similarity search
4. **Retriever** - Interface to fetch relevant documents
5. **RetrievalQA Chain** - Combine retrieval + LLM for answers

### Workflow:
Documents → Chunks → Embeddings → Vector Store
↓
Query → Embedding → Similarity Search → Retrieve Docs → LLM → Answer


### Chain Types:

| Chain Type | Speed | Cost | Best For |
|------------|-------|------|----------|
| **stuff** | Fast | Low | Small docs |
| **map_reduce** | Medium | High | Large docs, parallel |
| **refine** | Slow | High | Detailed answers |
| **map_rerank** | Medium | High | Best single answer |

### When to Use:

- **Internal knowledge bases** - Company wikis, documentation
- **Customer support** - Product manuals, FAQs
- **Research** - Academic papers, reports
- **E-commerce** - Product catalogs (like this example)